In [1]:
import pandas as pd
from typing import List
import json
import os
os.chdir("../../")

In [19]:
def create_directory(directory_path):
    if not os.path.exists(directory_path):
        os.makedirs(directory_path)
        print(f"Directory '{directory_path}' created.")
    else:
        pass
        # print(f"Directory '{directory_path}' already exists.")

def read_file(fname):
    output = []
    with open(fname) as f:
        for line in f:
            output.append(line.strip())
    return output

def read_jsonl(fname):
    output = []
    with open(fname, 'r', encoding='utf-8') as file:
        for line in file:
            output.append(json.loads(line))
    return output

def read_json(fname):
    with open(fname, 'r', encoding='utf-8') as file:
            output = json.load(file)
    return output

def get_instructions(lp, context_type, split="test"):
    source_lang = lp.split("-")[0].replace('pt','pt-br')
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    instructions = read_file(f"./instructions/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/instructions.txt")
    return instructions

def get_generations(lp, context_type,model_ver, split="test"):
    source_lang = lp.split("-")[0].replace('pt','pt-br')
    target_lang = lp.split("-")[1].replace('pt','pt-br')
    generations = read_file(f"./generations/{context_type}/mt/wmt24_chat_{split}.{source_lang}-{target_lang}/vllm/{model_ver}/generation.txt")
    return generations



In [23]:
WINDOW_LENGTHS = [2,6,10,15,20]
LANGS = ["en-de","en-fr","en-ko","en-nl","en-pt","de-en","fr-en","ko-en","nl-en","pt-en"]
MODEL_VERS = {"greedy-7b-ft-w-context":"TowerInstruct-7B-w-chat-empty-sys","greedy-7b-w-context":"TowerInstruct-7B-v0.2"}

data_dict = {}
for num_turns in WINDOW_LENGTHS:
    data_dict[num_turns] = {}
    for lang_dir in LANGS:
        data_dict[num_turns][lang_dir] = {}
        # get instructions  and generations for tower-chat
        ft_w_context_type = f"full_context_empty_sys_{num_turns}_turns"
        instr_ft_w_ctx = get_instructions(lang_dir,ft_w_context_type,"test")
        gen_ft_w_ctx = get_generations(lang_dir,ft_w_context_type,MODEL_VERS["greedy-7b-ft-w-context"],"test")
        # get instructions  and generations for tower-Instruct
        w_context_type = f"full_context_{num_turns}_turns"
        instr_w_ctx = get_instructions(lang_dir,w_context_type,"test")
        gen_w_ctx = get_generations(lang_dir,w_context_type,MODEL_VERS["greedy-7b-w-context"],"test")
        
        ## TODO: add here the ids of the files as indexes!
        data_list = [{"instruction-greedy-7b-ft-w-context":instr_ft,"greedy-7b-ft-w-context":gen_ft,"instrunction-greedy-7b-w-context":instr,"greedy-7b-w-context":gen} for instr_ft,gen_ft,instr,gen in zip(instr_ft_w_ctx,gen_ft_w_ctx,instr_w_ctx,gen_w_ctx)]
        data_dict[num_turns][lang_dir] = pd.DataFrame(data_list).reset_index()




In [24]:
for num_turns in data_dict.keys():
    for lang_dir in data_dict[num_turns].keys():
        dir_path = f"./alti_prep_data/fixed_context_window/{num_turns}_turns/"
        create_directory(dir_path)
        fname = os.path.join(dir_path,f'./test_{lang_dir}_datadf.json')
        data = data_dict[num_turns][lang_dir].to_dict(orient='records')
        with open(fname, 'w') as f:
            json.dump(data, f)